# 长鑫科技合理估值分析（三）结论、局限与进阶分析

涵盖报告第 5—7 节：初步结论、局限性、回归/事件研究/预测。


## 5. 初步结论

In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))
from plot_style import setup_theme
setup_theme()

DATA, CLEAN, OUTPUT = ROOT / "data", ROOT / "clean", ROOT / "output"
fin = pd.read_csv(CLEAN / "cxkj_financials_clean.csv")
stock = pd.read_csv(CLEAN / "stock_clean.csv", parse_dates=["date"])

rev2024 = fin.loc[(fin["metric"] == "revenue") & (fin["period"] == "2024"), "value_bn"].iloc[0]
rev2023 = fin.loc[(fin["metric"] == "revenue") & (fin["period"] == "2023"), "value_bn"].iloc[0]
loss_cum_bn = 408.57  # 招股书摘要：截至 2025-06-30 累计亏损

print("=== 关键事实（数据驱动）===")
print(f"2024 营业收入: {rev2024:.2f} 亿元")
print(f"2023→2024 营收增速: {(rev2024/rev2023-1)*100:.1f}%")
print(f"招股书披露累计亏损(至2025H1): 约 {loss_cum_bn} 亿元")
print(f"一级市场报道估值区间: 1282 — 1584 亿元")
print(f"拟募资规模: 295 亿元")

scenarios = []
for pe, lab in [(8, "保守"), (15, "中性"), (25, "乐观")]:
    scenarios.append({"情景": lab, "假设PE": pe, "基于2024收入(亿)": rev2024, "隐含市值(亿)": rev2024 * pe})
scenarios = pd.DataFrame(scenarios)
scenarios.to_csv(OUTPUT / "valuation_scenarios.csv", index=False, encoding="utf-8-sig")
display(scenarios)


**5.1 问题严重程度**：营收高速增长（2024 年 +166%）与累计亏损超 400 亿元并存，说明**产业战略意义重大，但财务安全边际仍低**。

**5.2 主要影响对象**：① 战略与财务投资者（Pre-IPO 股东）；② A 股存储/设备产业链（情绪与资金分流）；③ 地方政府与产业链配套（合肥集群）。

**5.3 待进一步研究**：单季 DRAM 价格、产能利用率、政府补助占比、发行定价 vs 同行 PS/EV-Sales、上市后解禁压力。


## 6. 局限性说明

| 类型 | 说明 |
|------|------|
| **数据** | 长鑫未上市，财务仅至申报稿；缺少季度分部毛利、产能利用率、市占率高频数据 |
| **方法** | 收入×PE 为示意性情景，未采用 DCF；CAPM 使用沪深300、日频收益，与科创板波动结构存在偏差 |
| **结论适用范围** | 适用于 IPO 前一级估值讨论与同业对比，**不构成投资建议**；上市后需更新发行价格与锁定期数据 |


## 7. 进一步分析

### 7.1 CAPM 回归（可比公司 Beta）

In [ ]:
idx = pd.read_csv(DATA / "index" / "index_sh000300.csv", parse_dates=["date"]).sort_values("date")
idx = idx.set_index("date")
idx["mkt_ret"] = idx["close"].pct_change()
STOCK_LIST = [
    ("603986", "兆易创新"), ("688008", "澜起科技"), ("300223", "北京君正"),
    ("688981", "中芯国际"), ("002371", "北方华创"), ("688012", "中微公司"),
    ("688396", "华润微"), ("603501", "韦尔股份"),
]
rows = []
for code, name in STOCK_LIST:
    g = stock[stock["code"] == code].set_index("date")
    m = g[["return"]].join(idx[["mkt_ret"]], how="inner").dropna()
    y = m["return"] - 0.02 / 252
    x = sm.add_constant(m["mkt_ret"] - 0.02 / 252)
    model = sm.OLS(y, x).fit()
    rows.append({"name": name, "beta": model.params["mkt_ret"], "alpha_ann": model.params["const"] * 252, "r2": model.rsquared})
capm = pd.DataFrame(rows)
capm.to_csv(OUTPUT / "capm_results.csv", index=False, encoding="utf-8-sig")
display(capm.sort_values("beta", ascending=False))

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(capm["name"], capm["beta"], color="#667eea")
ax.axvline(1, color="gray", linestyle="--")
ax.set_title("CAPM Beta（相对沪深300）"); ax.set_xlabel("Beta")
plt.tight_layout()
plt.savefig(OUTPUT / "fig_capm_beta.png", dpi=150)
plt.show()


**解读**：存储产业链公司 Beta 多高于 1，意味着系统性风险敞口较大；若市场下行，长鑫上市定价可能面临**折让压力**，除非发行窗口处于行业景气顶点。


### 7.2 一级市场估值区间（公开报道）

In [ ]:
labels = ["交易参考下限\n(2024.12)", "战略融资\n(2024.03)", "媒体报道上限\n(2026.01)"]
vals = [1282, 1400, 1584]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(labels, vals, color=["#90cdf4", "#4a90e2", "#2b6cb0"])
ax.set_xlabel("估值（亿元人民币）")
ax.set_title("一级市场估值区间（公开报道，非模型测算）")
for i, v in enumerate(vals):
    ax.text(v + 20, i, f"{v:,}亿", va="center")
plt.tight_layout()
plt.savefig(OUTPUT / "fig6_valuation_range.png", dpi=150)
plt.show()


**解读**：报道区间 1282—1584 亿元对应 2024 收入约 5.3—6.6 倍 **PS（市销率）**，高于传统制造、低于部分半导体设计龙头峰值，反映市场对**国产 DRAM 稀缺性**的溢价。本研究示意性 PE 情景（8/15/25 倍）对应市值约 1934—6045 亿元，需用**预期盈利年份**修正，不可直接对比。


### 7.3 收入线性预测（示意）

In [ ]:
rev = fin[fin["metric"] == "revenue"]
rev = rev[rev["period"].isin(["2022", "2023", "2024"])].copy()
rev["year"] = rev["period"].astype(int)
x, y = rev["year"].values, rev["value_bn"].values
coef = np.polyfit(x, y, 1)
future = [2025, 2026, 2027]
pred = np.polyval(coef, future)
fc = pd.DataFrame({"year": future, "revenue_forecast_bn": pred})
fc.to_csv(OUTPUT / "revenue_forecast.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, y, "o-", label="实际", color="#4a90e2")
ax.plot(future, pred, "s--", label="线性外推", color="#e74c3c")
ax.set_title("营业收入趋势与简单预测（亿元）"); ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "fig_revenue_forecast.png", dpi=150)
plt.show()
display(fc)


**解读**：线性外推仅作**敏感性示意**，未考虑 DRAM 价格均值回归；2025 年化收入可能高于直线预测，但盈利转正时间仍不确定。

---

**下一步**：运行 `scripts/analysis.py` 可一键复现全部图表；完整文字报告见 `output/summary_stats.md`。


In [ ]:
# 可选：一键运行全部分析脚本
import sys
sys.path.insert(0, str(ROOT / "scripts"))
from analysis import run_all
run_all()
